In [1]:
import os
import pandas as pd
import numpy as np
import plotnine as p9
# 注意：不再使用 patchworklib，改用 plotnine 原生的图形组合功能

In [2]:
import plotnine as p9

def create_mosaic_plot(data, metric, cmap='viridis'):
    """
    Create a mosaic plot using plotnine with faceting on the 'dataset' column.

    :param data: DataFrame with columns 'reduction_name', 'score_key', 'AUROC', and 'dataset'.
    :return: The plotnine plot object.
    """
    # Create the plot
    plot = (p9.ggplot(data, p9.aes(x='reduction_name', y='score_key', fill=metric)) +
            p9.geom_tile() +
            p9.theme_bw(base_size=11) +
            p9.facet_grid(' ~ dataset') +
            p9.scale_fill_cmap(limits=(0.5, 1), cmap_name=cmap) +
            p9.geom_text(p9.aes(label=metric), size=10, color='white', fontweight='bold') +
            p9.theme(axis_text_x=p9.element_text(angle=90, size=11),
                     strip_text_x=p9.element_text(size=11),
                     strip_background=p9.element_rect(colour="black", fill="#fdfff4"),
                     legend_text=p9.element_text(size=11),
                     axis_text_y=p9.element_text(size=11)) +
            p9.labs(x='', y='', fill=metric))

    return plot


Load Results and Concat

In [3]:
# list output files
file_paths = os.listdir(os.path.join('data', 'results'))
# keep only .csv files
results = [pd.read_csv(os.path.join('data', 'results', p)) for p in file_paths if p.endswith('.csv')]

In [4]:
results = pd.concat(results)

In [5]:
results['AUROC'] = results.groupby(['reduction_name', 'dataset', 'score_key'])['auroc'].transform('mean')
results['F1'] = results.groupby(['reduction_name', 'dataset', 'score_key'])['f1_score'].transform('mean')
results = results[['reduction_name', 'score_key', 'AUROC', 'F1', 'dataset']].drop_duplicates()

In [ ]:
remap_dict = {'lr_means':'CellPhoneDB',
              'expr_prod':'Product',
              'lr_logfc': 'logFC',
              'lrscore': 'SingleCellSignalR',
              'lr_probs': 'CellChat',
              'magnitude_rank':'Rank Aggregate',
              'inter_score': 'scSeqComm',
              'lr_gmeans': 'Geometric Mean',
              'hgatlink': 'HGATLink',
              }
# Prettify names
results['score_key'] = results['score_key'].map(remap_dict)
results['reduction_name'] = results['reduction_name'].replace({"mofa":"MOFA+", 'tensor':"Tensor-cell2cell"})

In [7]:
# mean rank per Method
score_avg = results.groupby(['score_key', 'reduction_name'])[['AUROC', 'F1']].agg('mean')
score_avg['dataset'] = "Score Average"
results = pd.concat([results, score_avg.reset_index()])
results['AUROC'] = results['AUROC'].round(2)
results['F1'] = results['F1'].round(2)

In [8]:
# to title
results['dataset'] = results['dataset'].replace({'Score Average': 'Average'})
results['dataset'] = pd.Categorical(results['dataset'], categories=['carraro', 'habermann', 'kuppe', 'velmeshev', 'reichart', 'Average'])
results['dataset'] = results['dataset'].str.title()

In [9]:
p1 = create_mosaic_plot(results, metric='AUROC')

In [10]:
p2 = create_mosaic_plot(results, metric='F1', cmap='cividis')

In [11]:
# 使用 matplotlib 将两个 plotnine 图形组合到同一页面
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import io
from PIL import Image

# 首先将每个 plotnine 图形渲染为图像
def plotnine_to_image(plot, width=10, height=4.5, dpi=300):
    """将 plotnine 图形转换为 PIL Image"""
    buf = io.BytesIO()
    plot.save(buf, format='png', width=width, height=height, dpi=dpi, verbose=False)
    buf.seek(0)
    return Image.open(buf)

# 转换两个图形为图像
img1 = plotnine_to_image(p1)
img2 = plotnine_to_image(p2)

# 创建组合图形
fig, axes = plt.subplots(2, 1, figsize=(10, 9))

# 移除 axes 的边框和刻度
for ax in axes:
    ax.axis('off')

# 显示图像
axes[0].imshow(img1)
axes[1].imshow(img2)

# 调整布局并保存
plt.tight_layout(pad=0.5)
fig.savefig(os.path.join('./', 'figures', 'classification.pdf'), 
            bbox_inches='tight', dpi=300)
plt.close(fig)

print("PDF saved with both plots combined on a single page.")

PDF saved with both plots combined on a single page.


In [13]:
# Move to Excel Sheet
p1.data.to_csv(os.path.join("./", "figures", "source", "ExtData_Fig5B&C.csv"))